# Homework Starter — Stage 05: Data Storage
Name: 
Date: 

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
!pip install numpy
!pip install pandas
!pip install pyarrow
!pip install python-dotenv

   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ---------- ----------------------------- 7.3/27.8 MB 41.2 MB/s eta 0:00:01
   ---------------------------------- ----- 23.9/27.8 MB 62.9 MB/s eta 0:00:01
   ---------------------------------------- 27.8/27.8 MB 57.0 MB/s  0:00:00


In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: e:\研究生\bootcamp\bootcamp_aixuan_liu\homework\homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> E:\研究生\bootcamp\bootcamp_aixuan_liu\homework\homework05\data\raw
PROC -> E:\研究生\bootcamp\bootcamp_aixuan_liu\homework\homework05\data\processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [5]:
SOURCE_FILE = pathlib.Path(
    'starter_data.csv'
)

df = pd.read_csv(
    SOURCE_FILE,
    parse_dates=['date']
)

display(
    df.head()
)

print(
    "\nShape:",
    df.shape
)

print(
    "\nData types:"
)

print(
    df.dtypes
)

,category,value,date
0,A,10,2025-08-01
1,B,15,2025-08-02
2,A,12,2025-08-03
3,B,18,2025-08-04
4,C,25,2025-08-05



Shape: (10, 3)

Data types:
category            object
value                int64
date        datetime64[ns]
dtype: object


In [6]:
print(
    "Columns:"
)

print(
    df.columns.tolist()
)

print(
    "\nMissing values:"
)

print(
    df.isna().sum()
)

print(
    "\nShape:"
)

print(
    df.shape
)

Columns:
['category', 'value', 'date']

Missing values:
category    0
value       0
date        0
dtype: int64

Shape:
(10, 3)


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [7]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# TODO: Save CSV
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
csv_path

# TODO: Save Parquet
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    pq_path = None
pq_path

WindowsPath('data/processed/sample_20260819-010105.parquet')

In [8]:
print(
    "CSV exists:",
    csv_path.exists()
)

print(
    "Parquet exists:",
    pq_path is not None
    and pq_path.exists()
)

CSV exists: True
Parquet exists: True


## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [9]:
def validate_loaded(
    original,
    reloaded
):

    checks = {

        'shape_equal':
            original.shape
            == reloaded.shape,

        'date_is_datetime':
            pd.api.types
            .is_datetime64_any_dtype(
                reloaded['date']
            )
            if 'date'
            in reloaded.columns
            else False,

        'value_is_numeric':
            pd.api.types
            .is_numeric_dtype(
                reloaded['value']
            )
            if 'value'
            in reloaded.columns
            else False,

        'category_is_text':
            pd.api.types
            .is_object_dtype(
                reloaded['category']
            )
            if 'category'
            in reloaded.columns
            else False,

        'required_columns_present':
            all(
                col in reloaded.columns
                for col in [
                    'category',
                    'value',
                    'date'
                ]
            )
    }

    checks[
        'all_passed'
    ] = all(
        checks.values()
    )

    return checks

In [10]:
df_csv = pd.read_csv(
    csv_path,
    parse_dates=['date']
)

display(
    df_csv.head()
)

csv_validation = validate_loaded(
    df,
    df_csv
)

csv_validation

,category,value,date
0,A,10,2025-08-01
1,B,15,2025-08-02
2,A,12,2025-08-03
3,B,18,2025-08-04
4,C,25,2025-08-05


{'shape_equal': True,
 'date_is_datetime': True,
 'value_is_numeric': True,
 'category_is_text': True,
 'required_columns_present': True,
 'all_passed': True}

In [11]:
if pq_path is not None:

    try:

        df_pq = pd.read_parquet(
            pq_path
        )

        display(
            df_pq.head()
        )

        parquet_validation = (
            validate_loaded(
                df,
                df_pq
            )
        )

        parquet_validation

    except Exception as e:

        print(
            "Parquet read failed:"
        )

        print(
            e
        )

,category,value,date
0,A,10,2025-08-01
1,B,15,2025-08-02
2,A,12,2025-08-03
3,B,18,2025-08-04
4,C,25,2025-08-05


In [12]:
validation_results = pd.DataFrame({
    'CSV': csv_validation,
    'Parquet': parquet_validation
})

display(
    validation_results
)

,CSV,Parquet
shape_equal,True,True
date_is_datetime,True,True
value_is_numeric,True,True
category_is_text,True,True
required_columns_present,True,True
all_passed,True,True


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [13]:
import typing as t
import pathlib


def detect_format(
    path: t.Union[
        str,
        pathlib.Path
    ]
):

    s = str(
        path
    ).lower()

    if s.endswith(
        '.csv'
    ):

        return 'csv'

    if (
        s.endswith('.parquet')
        or s.endswith('.pq')
        or s.endswith('.parq')
    ):

        return 'parquet'

    raise ValueError(
        'Unsupported format: '
        + s
    )

In [14]:
def write_df(
    df: pd.DataFrame,
    path: t.Union[
        str,
        pathlib.Path
    ]
):

    p = pathlib.Path(
        path
    )

    p.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    fmt = detect_format(
        p
    )

    if fmt == 'csv':

        df.to_csv(
            p,
            index=False
        )

    else:

        try:

            df.to_parquet(
                p,
                index=False
            )

        except ImportError as e:

            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

    print(
        "Written:",
        p
    )

    return p

In [15]:
def write_df(
    df: pd.DataFrame,
    path: t.Union[
        str,
        pathlib.Path
    ]
):

    p = pathlib.Path(
        path
    )

    p.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    fmt = detect_format(
        p
    )

    if fmt == 'csv':

        df.to_csv(
            p,
            index=False
        )

    else:

        try:

            df.to_parquet(
                p,
                index=False
            )

        except ImportError as e:

            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

    print(
        "Written:",
        p
    )

    return p

## Data Storage Summary

### Dataset

This notebook uses the provided `starter_data.csv` dataset.

The dataset contains the following columns:

- `category`: categorical/text data
- `value`: numeric data
- `date`: date/time data

### Storage Structure

The project uses two storage folders:

- `data/raw/` for raw CSV files
- `data/processed/` for processed Parquet files

The paths are configured using environment variables in `.env`:

- `DATA_DIR_RAW=data/raw`
- `DATA_DIR_PROCESSED=data/processed`

### Storage Formats

CSV is used for raw data because it is simple, portable, and human-readable.

Parquet is used for processed data because it preserves data types efficiently and is suitable for analytical workflows.

### Validation

After saving, both CSV and Parquet files are reloaded.

Validation checks include:

- matching dataframe shapes;
- confirming `date` remains datetime;
- confirming `value` remains numeric;
- confirming `category` remains text;
- confirming all required columns are present.

### Utilities

Three reusable utilities are implemented:

- `detect_format()` detects CSV or Parquet from the file suffix.
- `write_df()` writes the dataframe using the appropriate format and creates missing parent directories.
- `read_df()` loads the correct format based on the suffix.

The utilities also provide clear errors for unsupported file formats, missing files, and unavailable Parquet engines.

### Assumptions and Risks

- The provided dataset contains the expected `category`, `value`, and `date` columns.
- CSV does not automatically preserve datetime types, so dates are explicitly parsed when reloading CSV.
- Parquet requires a supported engine such as `pyarrow`.
- The `.env` paths are relative to the notebook working directory.